# Projeto: Execução Central - Validação com Great Expectations

### Objetivo  
Este notebook centraliza a execução das validações automáticas com **Great Expectations** aplicadas às tabelas das camadas *bronze* e *silver*. Ele serve como ponto único de orquestração, reunindo todas as funções, regras, dependências e rotinas críticas de validação de dados, geração de logs e controle de qualidade.

---

## Escopo

- Importação unificada de bibliotecas e utilitários fundamentais (`utils_run`)
- Definição de **validações estruturais e semânticas** para colunas obrigatórias e valores esperados.
- Geração de **logs de validação em formato JSON**, com timestamp ISO e salvamento automatizado no Blob Storage.
- Classificação de status com **indicadores visuais (🟢🟡🔴)** baseados no percentual de sucesso por tabela.
- Resumo dinâmico das validações recentes executadas (últimos 60 segundos).
- Modularização de validadores distintos para **camadas bronze e silver**, com regras específicas por tabela.
- Filtro de registros válidos e em quarentena para tratamento pós-validação.

---

## Histórico de Atualizações

| Data       | Responsável   | Versão | Descrição                                                                 |
|------------|---------------|--------|---------------------------------------------------------------------------|
| 24/05/2025 | Evelise Dias  | 1.0    | Estruturação inicial com centralização de funções, lógica de validação, logging e exibição de resumos.         |
| 25/05/2025 | Evelise Dias  | 1.1    | Inclusão de documentação formal.         |
| 26/05/2025 | Evelise Dias  | 1.2    | Ajuste de função de escrita.         |

In [0]:
%run ../utils_gx/utils_run

In [0]:
COLUNAS_OBRIGATORIAS = [
    "cidade", "data_previsao", "estado", "fonte",
    "precipitacao", "temperatura_maxima", "temperatura_minima", "data_carga"
]

In [0]:
def agora_iso():
    """
    Retorna o timestamp atual no formato ISO 8601 com fuso horário de São Paulo.

    Returns:
        str: Timestamp atual em formato ISO.
    """
    return datetime.now(ZoneInfo("America/Sao_Paulo")).isoformat()

In [0]:
def salvar_log(nome_tabela: str, resultado_validacao: dict, pasta_logs: str) -> str:
    """
    Salva o resultado da validação em um arquivo JSON com timestamp no nome.

    Args:
        nome_tabela (str): Nome da tabela validada.
        resultado_validacao (dict): Resultado do processo de validação.
        pasta_logs (str, opcional): Caminho da pasta onde o log será salvo. Default é "logs".

    Returns:
        str: Caminho completo do arquivo salvo.
    """
    timestamp = agora_iso()
    resultado_validacao["timestamp"] = timestamp
    safe_time = timestamp.replace(":", "-")
    caminho_log = os.path.join(pasta_logs, f"{nome_tabela}_{safe_time}.json")

    dbutils.fs.put(caminho_log, json.dumps(resultado_validacao, indent=4, ensure_ascii=False), overwrite=True)


    print(f"[LOG] Salvo em: {caminho_log}")
    return caminho_log


def calcular_percentual_sucesso(log):
    """
    Calcula o percentual de validações bem-sucedidas a partir do log.

    Args:
        log (dict): Log contendo estatísticas da validação.

    Returns:
        float: Percentual de expectativas bem-sucedidas.
    """
    stats = log.get("detalhes_validacao", {}).get("statistics", {})
    return (stats.get("successful_expectations", 0) / stats.get("evaluated_expectations", 1)) * 100



def classificar_status(percentual):
    """
    Classifica o percentual de sucesso em um status visual (emoji) e string formatada.

    Args:
        percentual (float): Percentual de sucesso da validação.

    Returns:
        tuple: Emoji de status e percentual formatado como string.
    """
    if percentual >= 80:
        return "🟢", "100%"
    elif percentual >= 60:
        return "🟡", f"{percentual:.1f}%"
    return "🔴", f"{percentual:.1f}%"



def pegar_logs_recentes(pasta_logs: str, segundos=60):
    """
    Retorna a lista de arquivos de log JSON modificados recentemente.

    Args:
        segundos (int, opcional): Tempo limite em segundos para considerar o log como recente. Default é 60.
        pasta (str, opcional): Pasta onde os logs estão armazenados. Default é "logs".

    Returns:
        list: Lista de caminhos dos arquivos JSON recentes.
    """
    limite = time.time() - segundos
    arquivos_recentes = []

    for arquivo in dbutils.fs.ls(pasta_logs):
        if arquivo.name.endswith(".json") and arquivo.modificationTime / 1000 >= limite:
            arquivos_recentes.append(arquivo.path)

    return arquivos_recentes


def exibir_resumo_validacoes(pasta_logs: str):
    """
    Exibe um resumo no terminal com o status das validações realizadas nos últimos 60 segundos.

    Args:
        pasta_logs (str): Caminho da pasta onde os logs estão armazenados. Padrão é 'logs'.

    Returns:
        None
    """
    arquivos = pegar_logs_recentes(pasta_logs)
    if not arquivos:
        print("[RESUMO] Nenhum log recente encontrado.")
        return

    print("="*70)
    print(" RESUMO DAS VALIDAÇÕES DE TABELAS".center(70))
    print("="*70)
    print(f"{'Tabela':<30} {'Status':<15} {'%':<10} {'Indicador'}")
    print("-"*70)

    for arq in arquivos:
        conteudo = dbutils.fs.head(arq)
        log = json.loads(conteudo)
        nome = log.get("tabela", "???")
        status = "SUCCESS" if log.get("sucesso") else "FAIL"
        percentual = calcular_percentual_sucesso(log)
        cor, pct = classificar_status(percentual)

        print(f"{nome:<30} {status:<15} {pct:<10} {cor}")
    print("="*70)

In [0]:
def criar_validador(contexto_path: str, df, tabela):
    """
    Cria e retorna um validador do Great Expectations para o DataFrame fornecido.
    Usa contexto persistente com lógica idempotente para execuções diárias.
    """
    try:
        context = gx.get_context(context_root_dir=contexto_path)
        datasource = context.data_sources.add_spark(f"gx_bronze_{tabela}")
    except Exception:
        context = gx.get_context()
        datasource = context.data_sources.get(f"gx_bronze_{tabela}")

    try:
        asset = datasource.get_asset(f"data_asset_bronze_{tabela}")
    except Exception:
        asset = datasource.add_dataframe_asset(name=f"data_asset_bronze_{tabela}")

    batch_name = f"batch_{datetime.now().strftime('%Y%m%d%H%M%S')}"
    batch_def = asset.add_batch_definition_whole_dataframe(batch_name)
    batch = batch_def.get_batch(batch_parameters={"dataframe": df})

    return context.get_validator(batch=batch)


def validar_dataframe(df, tabela_nome: str, contexto_path: str, pasta_logs: str):
    """
    Executa validações de colunas obrigatórias em um DataFrame e salva o log do processo.

    Args:
        df (DataFrame): DataFrame a ser validado.
        tabela_nome (str): Nome da tabela associada ao DataFrame.
        contexto_path (str): Caminho do diretório de contexto do Great Expectations.

    Returns:
        None
    """

    validator = criar_validador(contexto_path, df, tabela_nome)
    log = {
        "tabela": tabela_nome,
        "colunas_validadas": [],
        "colunas_ausentes": [],
        "erros_validacao_colunas": [],
    }
   
    validacoes = []

    for coluna in COLUNAS_OBRIGATORIAS:
        if coluna in df.columns:
            try:
                validator.expect_column_values_to_not_be_null(column=coluna)
                validacoes.append({"coluna": coluna})
                log["colunas_validadas"].append(coluna)
            except Exception as e:
                log["erros_validacao_colunas"].append({coluna: str(e)})
        else:
            log["colunas_ausentes"].append(coluna)

    try:
        resultado = validator.validate()
        log["validacao_executada"] = True
        log["sucesso"] = resultado.success

        log["detalhes_validacao"] = {
            "validacoes_aplicadas": validacoes,
            "statistics": resultado.statistics,
        }

    except Exception as e:
        log.update({
            "validacao_executada": False,
            "sucesso": False,
            "erro_execucao_validacao": str(e)
        })

    salvar_log(tabela_nome, log, pasta_logs)

    condicoes_validas = [col(c).isNotNull() for c in COLUNAS_OBRIGATORIAS if c in df.columns]
    condicoes_completas = reduce(lambda a, b: a & b, condicoes_validas)

    df_validado = df.filter(condicoes_completas)
    df_quarentena = df.filter(~condicoes_completas)

    return df_validado, df_quarentena

def listar_tabelas_existentes(catalogo, camada, tabelas):
    """
    Verifica se as tabelas existem no catálogo Spark e retorna as que foram encontradas.

    Args:
        catalogo (str): Nome do catálogo no Spark.
        camada (str): Nome da camada (schema) no catálogo.
        tabelas (list): Lista com os nomes das tabelas a verificar.

    Returns:
        list: Lista com nomes completos das tabelas encontradas.
    """
    encontradas = []
    for tabela in tabelas:
        nome = f"{catalogo}.{camada}.{tabela}"
        try:
            spark.table(nome)
            encontradas.append(nome)
            print(f"[OK] Tabela existe: {nome}")
        except:
            print(f"[ERRO] Não encontrada: {nome}")
    return encontradas

In [0]:
def criar_validador(contexto_path: str, df, tabela: str):
    """
    Cria e retorna um validador do Great Expectations para o DataFrame fornecido.

    Args:
        contexto_path (str): Caminho do diretório de contexto do Great Expectations.
        df (DataFrame): DataFrame a ser validado.

    Returns:
        Validator: Objeto validador do Great Expectations.
    """
    try:
        context = gx.get_context(context_root_dir=contexto_path)
        datasource = context.data_sources.add_spark(f"gx_silver_{tabela}")
    except Exception:
        context = gx.get_context()
        datasource = context.data_sources.get(f"gx_silver_{tabela}")

    try:
        asset = datasource.get_asset(f"data_asset_silver_{tabela}")
    except Exception:
        asset = datasource.add_dataframe_asset(name=f"data_asset_silver_{tabela}")

    batch_name = f"batch_{datetime.now().strftime('%Y%m%d%H%M%S')}"
    batch_def = asset.add_batch_definition_whole_dataframe(batch_name)
    batch = batch_def.get_batch(batch_parameters={"dataframe": df})

    return context.get_validator(batch=batch)

def validar_dataframe_silver(df, tabela_nome: str, contexto_path: str, pasta_logs: str):
    regras_por_tabela = {
        "default": {
            "colunas_numericas": {
                "temp_min": {"min": -50, "max": 60},
                "temp_max": {"min": -50, "max": 60},
                "prob_precipitacao": {"min": 0, "max": 100},
                "precipitacao": {"min": 0, "max": 200},
            },
            "colunas_obrigatorias": ["temp_min", "temp_max", "prob_precipitacao", "precipitacao"],
        },
        "inpe": {
            "colunas_numericas": {
                "temp_min": {"min": -50, "max": 60},
                "temp_max": {"min": -50, "max": 60},
                "prob_precipitacao": {"min": 0, "max": 100},
            },
            "colunas_obrigatorias": ["temp_min", "temp_max", "prob_precipitacao"],
        },
        "chuvas_criticas": {
            "colunas_numericas": {
                "vol_max_chuva": {"min": 0, "max": 100},
            },
            "colunas_obrigatorias": ["vol_max_chuva", "nom_municipio", "sig_uf"],
        }
    }

    if tabela_nome == "grupo01.silver.inpe":
        regras = regras_por_tabela["inpe"]
    elif tabela_nome == "grupo01.silver.chuvas_criticas":
        regras = regras_por_tabela["chuvas_criticas"]
    else:
        regras = regras_por_tabela["default"]

    colunas_numericas = regras["colunas_numericas"]
    colunas_obrigatorias = regras["colunas_obrigatorias"]

    colunas_ausentes = [col for col in colunas_obrigatorias if col not in df.columns]

    if colunas_ausentes:
        raise ValueError(f"Colunas obrigatórias ausentes no DataFrame: {colunas_ausentes}")

    validator = criar_validador(contexto_path, df, tabela_nome)
    validacoes = []

    log = {
        "tabela": tabela_nome,
        "colunas_ausentes": colunas_ausentes,
        "erros_validacao_colunas": [],
        "validacoes_personalizadas": [],
    }

    # Aplicar validações numéricas estritas
    for coluna, limites in colunas_numericas.items():
        if coluna in df.columns:
            validator.expect_column_values_to_be_between(
                column=coluna,
                min_value=limites["min"],
                max_value=limites["max"],
                mostly=1.0,
            )
            validacoes.append({"coluna": coluna})
            log["validacoes_personalizadas"].append(
                f"{coluna} entre {limites['min']} e {limites['max']}"
            )
            
        else:
            pass

    # Validação timestamp 'dt_carga', só se existir explicitamente
    if "dt_carga" in df.columns:
        if isinstance(df.schema["dt_carga"].dataType, TimestampType):
            log["validacoes_personalizadas"].append("dt_carga possui timestamp válido")
        else:
            log["validacoes_personalizadas"].append("dt_carga existe, mas não é timestamp")

    try:
        resultado = validator.validate()
        log["validacao_executada"] = True
        log["sucesso"] = resultado.success

        log["detalhes_validacao"] = {
            "validacoes_aplicadas": validacoes,
            "statistics": resultado.statistics,
        }

    except Exception as e:
        log.update({
            "validacao_executada": False,
            "sucesso": False,
            "erro_execucao_validacao": str(e)
        })

    salvar_log(tabela_nome, log, pasta_logs)

    if tabela_nome == "inpe":
        condicoes_validas = [col(c).isNotNull() for c in ["temp_min", "temp_max"]]
    else:
        condicoes_validas = [col(c).isNotNull() for c in colunas_obrigatorias]

    condicoes_completas = reduce(lambda a, b: a & b, condicoes_validas)

    df_validado = df.filter(condicoes_completas)
    df_quarentena = df.filter(~condicoes_completas)

    return df_validado, df_quarentena



def listar_tabelas_existentes(catalogo, camada, tabelas):
    """
    Verifica se as tabelas existem no catálogo Spark e retorna as que foram encontradas.

    Args:
        catalogo (str): Nome do catálogo no Spark.
        camada (str): Nome da camada (schema) no catálogo.
        tabelas (list): Lista com os nomes das tabelas a verificar.

    Returns:
        list: Lista com nomes completos das tabelas encontradas.
    """
    encontradas = []
    for tabela in tabelas:
        nome = f"{catalogo}.{camada}.{tabela}"
        try:
            spark.table(nome)
            encontradas.append(nome)
            print(f"[OK] Tabela existe: {nome}")
        except:
            print(f"[ERRO] Não encontrada: {nome}")
    return encontradas

In [0]:
def validar_tabelas_bronze(catalogo: str, camada: str, tabelas: list, contexto_path, pasta_logs: str):
    """
    Executa validação nas tabelas da camada bronze, retorna os DataFrames validados e em quarentena.

    Returns:
        dict: {
            "nome_tabela": {
                "df_validado": DataFrame,
                "df_quarentena": DataFrame
            },
            ...
        }
    """
    resultados = {}
    existentes = listar_tabelas_existentes(catalogo, camada, tabelas)

    if not existentes:
        raise ValueError("[ERRO] Nenhuma tabela encontrada na camada especificada.")

    for tabela in existentes:
        try:
            df = spark.table(tabela)
            df_validado, df_quarentena = validar_dataframe(df, tabela, contexto_path, pasta_logs)

            resultados[tabela] = {
                "df_validado": df_validado,
                "df_quarentena": df_quarentena
            }

        except Exception as e:
            print(f"[FALHA] {tabela}: {str(e)}")
            resultados[tabela] = {
                "df_validado": None,
                "df_quarentena": None,
                "erro": str(e)
            }

    return resultados

In [0]:
def validar_tabelas_silver(catalogo: str, camada: str, tabelas: list, contexto_path, pasta_logs: str):
    """
    Executa validação nas tabelas da camada bronze, retorna os DataFrames validados e em quarentena.

    Returns:
        dict: {
            "nome_tabela": {
                "df_validado": DataFrame,
                "df_quarentena": DataFrame
            },
            ...
        }
    """
    resultados = {}
    existentes = listar_tabelas_existentes(catalogo, camada, tabelas)

    if not existentes:
        raise ValueError("[ERRO] Nenhuma tabela encontrada na camada especificada.")

    for tabela in existentes:
        try:
            df = spark.table(tabela)
            nome_tabela = tabela.split(".")[-1]
            df_validado, df_quarentena = validar_dataframe_silver(df, tabela, contexto_path, pasta_logs)

            resultados[tabela] = {
                "df_validado": df_validado,
                "df_quarentena": df_quarentena
            }

        except Exception as e:
            print(f"[FALHA] {tabela}: {str(e)}")
            resultados[tabela] = {
                "df_validado": None,
                "df_quarentena": None,
                "erro": str(e)
            }

    return resultados

In [0]:
def montar_mensagem(pasta_logs: str):
    arquivos = dbutils.fs.ls(pasta_logs)
    arquivos_json = [f for f in arquivos if f.path.endswith(".json")]
    arquivos_ordenados = sorted(arquivos_json, key=lambda f: f.modificationTime, reverse=True)
    arquivos_recentes = [f.path for f in arquivos_ordenados[:1]]  

    if not arquivos_recentes:
        return "Nenhum log encontrado para a última execução."

    largura_tabela = 100
    linhas = []
    linhas.append("Alerta de validações recentes.")
    linhas.append("")
    linhas.append("Foram detectadas execuções de validação nas últimas horas.")
    linhas.append("Confira o resumo abaixo para identificar possíveis falhas ou alertas críticos.")
    linhas.append("")
    titulo = "RESUMO DAS VALIDAÇÕES DE TABELAS"
    linhas.append(titulo.center(largura_tabela))  
    linhas.append(f"{'Tabela':<25} {'Status':<15} {'%':<10} {'Indicador':<10}")
    linhas.append("-" * largura_tabela)

    for caminho in arquivos_recentes:
        try:
            conteudo = spark.read.text(caminho).selectExpr("collect_list(value) as linhas").first()["linhas"]
            json_str = "\n".join(conteudo)
            log = json.loads(json_str)
        except Exception as e:
            linhas.append(f"{caminho:<25} ERRO --- Arquivo inválido")
            continue

        tabela = log.get("tabela", "desconhecida")
        status = "SUCCESS" if log.get("sucesso", False) else "FAIL"
        percentual = calcular_percentual_sucesso(log)
        cor, percentual_fmt = classificar_status(percentual)
        linha = f"{tabela:<25} {status:<15} {percentual_fmt:<10} {cor:<10}"
        linhas.append(linha)

    linhas.append("=" * 84)  
    return "\n".join(linhas)


In [0]:
def enviar_alerta_teams(mensagem, webhook_url):
    headers = {"Content-Type": "application/json"}
    payload = {
        "text": mensagem
    }
    response = requests.post(webhook_url, json=payload, headers=headers)
    response.raise_for_status() 
    if response.status_code == 200:
        print("Alerta enviado com sucesso!")
    else:
        print(f"Erro ao enviar alerta: {response.status_code} - {response.text}")

In [0]:
def escreve_com_merge_delta(
    dataframe: DataFrame,
    caminho_tabela_delta: str,
    campos_chave: List[str],
    alias_origem: str = "origem",
    alias_destino: str = "destino"
) -> None:
    """
    Realiza um upsert (merge) entre um DataFrame e uma tabela Delta salva em um caminho (sem catálogo).
    """

    spark = SparkSession.builder.getOrCreate()

    if not DeltaTable.isDeltaTable(spark, caminho_tabela_delta):
        print(f"[INFO] Tabela Delta não existe no caminho {caminho_tabela_delta}. Criando nova tabela.")
        dataframe.write.format("delta").mode("overwrite").save(caminho_tabela_delta)
        print("[SUCESSO] Nova tabela criada com sucesso.")
        return

    try:
        deltaTable = DeltaTable.forPath(spark, caminho_tabela_delta)
    except Exception as e:
        raise Exception(f"[ERRO] A tabela no caminho '{caminho_tabela_delta}' existe, mas não é uma Delta Table válida.\n{e}")

    condicoes = " AND ".join([f"{alias_origem}.{campo} = {alias_destino}.{campo}" for campo in campos_chave])

    # Atualiza todos os campos com os da origem
    update_dict = {col: f"{alias_origem}.{col}" for col in dataframe.columns}

    print(f"[INFO] Iniciando MERGE com base nos campos-chave: {campos_chave}")
    deltaTable.alias(alias_destino).merge(
        source=dataframe.alias(alias_origem),
        condition=condicoes
    ).whenMatchedUpdate(set=update_dict) \
     .whenNotMatchedInsertAll() \
     .execute()

    print("[SUCESSO] MERGE concluído com sucesso.")
